# Clustering: structure without fitting the known labels

**Question.** Do recurring wine-chemistry profiles appear when the algorithm is not shown cultivar labels?

We will standardize the declared chemistry features inside the package, fit repeated $k$-means solutions, inspect internal separation and stability, visualize profiles, and only then use cultivar as limited external audit evidence. A cluster is an algorithmic partition, not automatically a natural kind.

**What this example models.** An unsupervised method begins with a real structural question and a definition of useful structure. Separation, stability, domain meaning, and sensitivity answer different parts of that question.

**Your project version.** Define what a useful grouping would mean before fitting, justify the representation and distance, implement and test the method, assess stability across seeds and samples, characterize cluster profiles, and avoid treating a partition as a discovered natural law.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from dsm_project.config import load_config
from dsm_project.data import load_project_dataset
from dsm_project.experiments import run_clustering_example, run_pca_example

config = load_config("configs/example.toml")
dataset = load_project_dataset(config)
task = dataset.tasks["unsupervised"]
frame = dataset.observations

result = run_clustering_example(dataset, config, clusters=3)
print("Silhouette:", result.silhouette)
print("Seed-to-seed adjusted Rand index:", result.stability_ari)
print("Cultivar audit adjusted Rand index:", result.external_audit_ari)

## The $k$-means objective

Given standardized vectors $x_i$, assignments $c_i\in\{1,\ldots,k\}$, and centroids $\mu_j$, $k$-means minimizes within-cluster squared distance,

$$
\sum_{i=1}^{n}\lVert x_i-\mu_{c_i}\rVert_2^2.
$$

Alternating assignment and centroid updates decreases this objective but can stop at different local solutions. The number of clusters is supplied by the analyst. Scaling defines the geometry. Neither the objective nor a colorful plot proves that the partition is scientifically useful.

In [ ]:
features = list(result.feature_names)
clustered = frame[features].assign(cluster=result.labels)
raw_profiles = clustered.groupby("cluster").mean()
standardized_profiles = (raw_profiles - frame[features].mean()) / frame[features].std()

fig, ax = plt.subplots(figsize=(11, 4))
image = ax.imshow(standardized_profiles, aspect="auto", cmap="coolwarm", vmin=-2, vmax=2)
ax.set_yticks(range(len(standardized_profiles)), labels=standardized_profiles.index)
ax.set_xticks(range(len(features)), labels=features, rotation=70, ha="right")
ax.set_xlabel("chemistry feature")
ax.set_ylabel("cluster")
ax.set_title("Cluster profiles as standardized deviations from the sample mean")
fig.colorbar(image, ax=ax, label="standardized profile")
plt.tight_layout();

## Separation, stability, and external audit answer different questions

- Silhouette summarizes cohesion and separation under the chosen distance.
- Seed-to-seed adjusted Rand index asks whether random initialization changes assignments.
- External adjusted Rand index compares with recorded cultivar after fitting.

High agreement with cultivar can be interesting but does not retroactively make cultivar the unsupervised training target. Stability can also preserve a consistently unhelpful partition. Domain utility needs its own evidence.

In [ ]:
representation = run_pca_example(dataset)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(
    representation.coordinates[:, 0],
    representation.coordinates[:, 1],
    c=result.labels,
    cmap="viridis",
    alpha=0.8,
)
axes[0].set(title="Projection colored by fitted cluster", xlabel="PC1", ylabel="PC2")

audit = axes[1].scatter(
    representation.coordinates[:, 0],
    representation.coordinates[:, 1],
    c=frame["cultivar"],
    cmap="plasma",
    alpha=0.8,
)
axes[1].set(title="Same projection colored by held-out audit label", xlabel="PC1", ylabel="PC2")
plt.tight_layout();

## Conclusion, limits, and student implementation

A project should test sensitivity to scaling, feature representation, $k$, initialization, sample perturbation, outliers and alternative clustering assumptions. Define what a useful group means before optimizing a convenient index, and inspect individual assignments rather than only centroids.

A student implementation should test assignment distances, centroid updates, empty-cluster behavior, convergence, reproducible randomness and objective monotonicity. Keep any known labels out of fitting and selection unless the project explicitly changes to a supervised task.